# Lecture 7 · Notebook 1 — A ReAct agent loop in 40 lines
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture07/lecture07_1_react_agent_loop.ipynb)

**Companion to [Chapter 2 of the video lecture](https://nlp-course-cuhksz.github.io/lecture07.html#p2-s31)** (slide 31, *The agent loop*), [Chapter 3](https://nlp-course-cuhksz.github.io/lecture07.html#p3-s41)
(slide 41, *ReAct*) and [Chapter 4](https://nlp-course-cuhksz.github.io/lecture07.html#p4-s52) (slide 52, *Tool calling, step by step*). The lecture claims that the core of
Claude Code, Codex and Manus is a loop of about five lines, and that everything else is the **harness**: which tools exist, what may
run, what goes back into the context, and when to stop. In this notebook you will:

1. declare tools the way a harness does (name, description, JSON schema),
2. write the five-line loop and run it with a **scripted mock LLM**, so you can see every message,
3. plug in a **real small LLM** (Qwen2.5-0.5B-Instruct, about 1 GB) that writes its own tool calls,
4. add three pieces of harness: a **permission gate**, **errors that teach**, and a **step budget**, and
5. measure the same model in two harnesses, the *harness tax* from slide 81 in miniature.

Runs on a free CPU runtime in about 15 minutes, two thirds of it the experiment in section 5; a T4 GPU (*Runtime → Change runtime
type*) brings it down to about 5 minutes but is not needed.

In [ ]:
!pip -q install -U transformers accelerate

## 1 · Tools, as the harness declares them
Slide 52, step one: the harness tells the model which tools exist, with a **name**, a **description** and a **JSON schema** for the
arguments. The model never sees the Python code behind them, only these declarations. Our three tools are deliberately small and
deterministic: a weather lookup (mock data), a calculator, and a search over a five-entry "encyclopedia".

In [ ]:
import json, ast, operator, re

WEATHER = {"shenzhen": "28°C, cloudy", "beijing": "9°C, sunny", "shanghai": "17°C, light rain", "hong kong": "26°C, cloudy"}
WIKI = {
    "the society of mind": "The Society of Mind is a 1986 book by Marvin Minsky about how intelligence emerges from many simple agents.",
    "marvin minsky": "Marvin Minsky (born August 9, 1927) was an American cognitive scientist and co-founder of the MIT AI Lab.",
    "eliza": "ELIZA is a 1966 chatbot by Joseph Weizenbaum that imitated a psychotherapist with pattern-matching rules.",
    "react": "ReAct (2022) is a prompting method in which a language model interleaves reasoning traces and actions.",
    "swe-bench": "SWE-bench (2023) asks a model to fix real GitHub issues; a patch counts if the repository's tests pass.",
}

def get_weather(city: str) -> str:
    return WEATHER.get(city.strip().lower(), f"No weather data for '{city}'.")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}
def calculator(expression: str) -> str:
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("unsupported expression")
    return str(ev(ast.parse(expression, mode="eval").body))      # raises on bad input; the harness catches it

def search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in WIKI.items() if key in q or q in key]
    return " ".join(hits) if hits else f"No results for '{query}'."

def send_email(to: str, body: str) -> str:
    return f"Email sent to {to}."          # a tool with a side effect: the harness must ask first

TOOLS = [
    {"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a city.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "City name, e.g. Shenzhen"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "calculator", "description": "Evaluate an arithmetic expression with + - * / ** and parentheses.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "e.g. (1986 - 1927) * 2"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "search", "description": "Search a small encyclopedia. Use a short topic name as the query.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "e.g. Marvin Minsky"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "send_email", "description": "Send an email on the user's behalf.",
        "parameters": {"type": "object", "properties": {"to": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "body"]}}},
]
FUNCS = {"get_weather": get_weather, "calculator": calculator, "search": search, "send_email": send_email}
print(json.dumps(TOOLS[0], indent=1))

## 2 · The loop, and a harness that runs the tools
Slide 31 in code. `llm(messages, tools)` returns a reply with optional `tool_calls`. If there are none, the reply is final.
Otherwise the **harness** runs each call and appends the observation. The key point in red on slide 52: *the model never runs
anything*. `run_tool` is where the harness decides whether and how to run a call; for now it simply runs it.

In [ ]:
def run_tool(call):
    return str(FUNCS[call["name"]](**call["arguments"]))

def agent_loop(llm, task, system="You are a helpful assistant. Use the tools when they help.", max_steps=8, verbose=True):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    for step in range(max_steps):
        reply = llm(messages, TOOLS)                                   # think: choose an action (or answer)
        messages.append({"role": "assistant", "content": reply["content"],
                         "tool_calls": [{"type": "function", "function": c} for c in reply["tool_calls"]]})
        if not reply["tool_calls"]:                                    # final answer: stop
            if verbose: print(f"[final]   {reply['content']}")
            return reply["content"], messages
        for call in reply["tool_calls"]:                               # act, then observe
            obs = run_tool(call)
            messages.append({"role": "tool", "name": call["name"], "content": obs})
            if verbose:
                if reply["content"].strip(): print(f"[thought] {reply['content'].strip()}")
                print(f"[action]  {call['name']}({json.dumps(call['arguments'], ensure_ascii=False)})")
                print(f"[observe] {obs}")
    if verbose: print("[stopped] step budget exhausted")
    return None, messages

### A scripted "LLM" first
Before using a real model, we replay a fixed ReAct trace for a two-hop question (slide 41: *Thought, Action, Observation*, repeated).
The mock looks only at how many tool results it has seen. It shows the mechanics without any randomness.

In [ ]:
SCRIPT = [
    ("I need the publication year of the book first.", {"name": "search", "arguments": {"query": "The Society of Mind"}}),
    ("It was published in 1986 by Marvin Minsky. Now I need his birth year.", {"name": "search", "arguments": {"query": "Marvin Minsky"}}),
    ("Born 1927. His age in 1986 is 1986 - 1927, or one less if the book came out before his birthday.", {"name": "calculator", "arguments": {"expression": "1986 - 1927"}}),
    ("Minsky was about 59 (58 or 59, depending on the month) when The Society of Mind was published in 1986.", None),
]

def mock_llm(messages, tools):
    k = sum(m["role"] == "tool" for m in messages)
    thought, call = SCRIPT[k]
    return {"content": thought, "tool_calls": [call] if call else []}

answer, trace = agent_loop(mock_llm, "How old was the author of 'The Society of Mind' when it was published?")
print(len(trace), "messages in the context at the end")

Every observation goes back into `messages`, so the context grows with each step: the reason context engineering (Chapter 6) exists.

## 3 · A real model writes the tool calls
Now the same loop with a real open model. `apply_chat_template(..., tools=TOOLS)` renders the tool schemas into the prompt in the
format the model was post-trained on; Qwen2.5 answers with `<tool_call>{"name": ..., "arguments": ...}</tool_call>` blocks, which we
parse back into structured calls. This parser is part of the harness too.

The 0.5B model is tiny, so expect mistakes. On a GPU you can switch to `Qwen/Qwen2.5-1.5B-Instruct` or `Qwen/Qwen2.5-3B-Instruct`
and watch the mistakes become rarer.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"        # on a GPU: "Qwen/Qwen2.5-1.5B-Instruct" or "Qwen/Qwen2.5-3B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
lm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print("device:", device, "| parameters:", round(sum(p.numel() for p in lm.parameters()) / 1e6), "M")

def parse_reply(text):
    calls, bad = [], []
    for blob in re.findall(r"<tool_call>(.*?)</tool_call>", text, re.S):
        try:
            c = json.loads(blob.strip())
            args = c.get("arguments", {})
            calls.append({"name": c["name"], "arguments": json.loads(args) if isinstance(args, str) else args})
        except Exception:
            bad.append(blob.strip())
    content = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S).strip()
    return {"content": content, "tool_calls": calls, "unparsed": bad}

def make_llm(temperature=0.0, seed=0, max_new_tokens=160):
    g = torch.Generator(device="cpu").manual_seed(seed)
    def llm(messages, tools):
        prompt = tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False)
        ids = tok(prompt, return_tensors="pt").to(device)
        kw = dict(do_sample=True, temperature=temperature, top_p=0.9) if temperature > 0 else dict(do_sample=False, temperature=None, top_p=None, top_k=None)
        if temperature > 0: torch.manual_seed(int(torch.randint(0, 2**31 - 1, (1,), generator=g)))
        with torch.no_grad():
            out = lm.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **kw)
        return parse_reply(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
    return llm

llm = make_llm()
for task in ["What's the weather in Shenzhen right now? Should I take an umbrella?",
             "What is 1234 * 5678 + 91?"]:
    print("\nTASK:", task)
    agent_loop(llm, task)

Look at the rendered prompt once: the tool declarations from section 1 are just text in the system message. "Function calling" is a
format the model learned in post-training, not a separate mechanism.

In [ ]:
prompt = tok.apply_chat_template([{"role": "user", "content": "Weather in Beijing?"}], tools=TOOLS[:1], add_generation_prompt=True, tokenize=False)
print(prompt)

## 4 · Three pieces of harness
The loop above trusts the model completely. A real harness (slides 52, 118 and 122) adds at least three things:

* a **permission gate**: risky tools (here `send_email`) need human approval; the model only *asks*,
* **errors that teach**: if a tool fails, return an error text that says how to fix the call, instead of crashing (principle 6 on slide 131),
* a **step budget**: `max_steps` already stops a looping agent; we also stop on calls to unknown tools.

In [ ]:
APPROVED = set()            # tools the user has approved for this session; send_email is not in it

def run_tool(call):
    name, args = call["name"], call["arguments"]
    if name not in FUNCS:
        return f"Error: unknown tool '{name}'. Available tools: {', '.join(FUNCS)}."
    if name == "send_email" and name not in APPROVED:
        return "Denied: sending email needs the user's approval. Tell the user what you would send instead."
    try:
        return str(FUNCS[name](**args))
    except TypeError:
        return f"Error: wrong arguments {list(args)} for {name}. Expected: {list(TOOLS[[t['function']['name'] for t in TOOLS].index(name)]['function']['parameters']['properties'])}."
    except Exception as e:
        hint = " Use only numbers, + - * / ** and parentheses, e.g. 2**10 or (3+4)*5." if name == "calculator" else ""
        return f"Error: {type(e).__name__}: {e}.{hint}"

# the scripted model makes two typical mistakes; watch the harness turn them into feedback
BAD_SCRIPT = [
    ("", {"name": "calculator", "arguments": {"expression": "2^10 × 3"}}),
    ("The calculator wants ** for powers and * for times.", {"name": "calculator", "arguments": {"expression": "2**10 * 3"}}),
    ("Now email the result.", {"name": "send_email", "arguments": {"to": "ta@example.com", "body": "3072"}}),
    ("I am not allowed to send email without approval. The answer is 3072; here is the email I would send: 'Result: 3072'.", None),
]
def bad_mock(messages, tools):
    k = sum(m["role"] == "tool" for m in messages)
    thought, call = BAD_SCRIPT[k]
    return {"content": thought, "tool_calls": [call] if call else []}

agent_loop(bad_mock, "Compute 2 to the 10th, times 3, and email it to my TA.");

Now the real model with the new harness, on a task that tempts it to send an email:

In [ ]:
agent_loop(llm, "Look up what ELIZA is and email a one-line summary to ta@example.com.");

## 5 · Same model, two harnesses
Slides 80–81: the same weights can score very differently in different harnesses. We test that in miniature. Five small tasks, each
with an **outcome check** (slide 124: grade the outcome, not the agent's claim). Harness **A** is bare: vague tool descriptions and
unhelpful errors. Harness **B** is the one we just built: clear descriptions and errors that teach. We sample at temperature 0.7 and
run every task several times, so you also get **pass@1** and **pass^k** (Notebook 2 explains them).

This is the slow cell: 30 agent runs, roughly 10 minutes on a CPU runtime and 2–3 minutes on a T4 GPU (lower `TRIALS` to 2 if you are in a hurry). Small models are noisy, so your numbers will differ from run to run.

In [ ]:
import copy, numpy as np

TASKS = [
    ("What's the weather in Shanghai?", lambda a: "17" in a and "rain" in a.lower()),
    ("What is 4321 * 1234?", lambda a: "5332114" in a.replace(",", "")),
    ("What is 2 to the power of 20?", lambda a: "1048576" in a.replace(",", "")),
    ("In what year was the chatbot ELIZA created?", lambda a: "1966" in a),
    ("What is (1986 - 1927) * 12?", lambda a: "708" in a),
]

TOOLS_B = TOOLS
TOOLS_A = copy.deepcopy(TOOLS)
for t in TOOLS_A:                                      # harness A: vague descriptions, cryptic argument docs
    t["function"]["description"] = t["function"]["name"]
    for p in t["function"]["parameters"]["properties"].values():
        p.pop("description", None)

def run_tool_A(call):                                  # harness A: errors that do not teach
    try:
        return str(FUNCS[call["name"]](**call["arguments"]))
    except Exception:
        return "Error."
run_tool_B = run_tool

def evaluate(tools, runner, trials=3, seed=0):
    global TOOLS, run_tool
    saved = TOOLS, run_tool
    TOOLS, run_tool = tools, runner
    results = np.zeros((len(TASKS), trials), dtype=int)
    try:
        for i, (task, check) in enumerate(TASKS):
            for j in range(trials):
                ans, _ = agent_loop(make_llm(temperature=0.7, seed=seed + 100 * i + j, max_new_tokens=96), task, max_steps=4, verbose=False)
                results[i, j] = int(ans is not None and check(ans))
            print(f"  task {i + 1}/{len(TASKS)}: {results[i].sum()}/{trials}", flush=True)
    finally:
        TOOLS, run_tool = saved
    return results

TRIALS = 3
res = {}
for name, t, r in [("A (bare)", TOOLS_A, run_tool_A), ("B (harnessed)", TOOLS_B, run_tool_B)]:
    print("harness", name); res[name] = evaluate(t, r, TRIALS)
for name, r in res.items():
    print(f"harness {name:14s} per-task successes {r.sum(1).tolist()} of {TRIALS} | pass@1 = {r.mean():.2f} | "
          f"pass^{TRIALS} = {np.mean(r.min(1)):.2f} | pass@{TRIALS} = {np.mean(r.max(1)):.2f}")

In our test run on CPU, harness A reached pass@1 = 0.20 and harness B 0.67. Reading the traces shows why: with vague
descriptions the model invented tools such as `multiply_numbers`, and the bare `Error.` gave it nothing to recover from. Harness B's
remaining failures were the model's own, for example calling `get_weather` to answer a question about ELIZA.

Read the result the way slide 81 suggests: *a score belongs to a model–harness pair*. With five tasks and three trials the error bars
are wide (Notebook 2, section 5, shows how to check whether a difference is real), so treat this as a demonstration, not a result.

## Exercises
1. **Find the failure.** Re-run one failed task with `verbose=True`. Is the mistake in the model's call, in the parser, or in the
   harness? Fix the harness (not the task) and re-measure, the steering loop from slide 94.
2. **A bigger model.** On a T4 GPU, set `MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"`. How much of the gap between harness A and B is left?
   That is the bitter lesson of slides 127–129 in miniature.
3. **A new tool.** Add a `unit_convert(value, from_unit, to_unit)` tool with a JSON schema and a good docstring-like description.
   Ask *"How many feet is the height of a 828 m tower?"* and check that the model finds and uses it.
4. **Thoughts.** Change the system prompt to ask for one sentence starting with `Thought:` before every tool call (ReAct, slide 41).
   Does the success rate change?

**Next:** [Notebook 2 — pass@k vs. pass^k]({colab(N2)})